# Week 6: 期中复习交互式教程 (Midterm Review Interactive Tutorial)

> **课程:** CST8509 Reinforcement Learning | **主题:** Midterm Review (Weeks 1-5)
>
> **核心目标：** 通过交互式练习，巩固 Weeks 1-5 所有考试要点
>
> **学习路径：** RL 基础 → MDP & Bellman → Q-Learning 实战 → Gymnasium API → SB3 & DQN

---

## 0. 环境准备 (Setup)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 中文字体设置（如果可用）| CJK font setup
try:
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

print("✅ 环境准备完成 | Setup complete")

## 1. RL 基础回顾 (RL Fundamentals Review) — Week 1

### 核心问题
> "什么是强化学习？它和监督学习/无监督学习有什么不同？"

### Agent-Environment 交互循环 (Midterm Slide 5 必考: 画图题)

```
Agent (智能体)          Environment (环境)
  │                         │
  │──── Action a_t ────────>│
  │                         │
  │<── Reward R_{t+1} ──────│
  │<── State S_{t+1} ───────│
  │                         │
  └─── 重复 (Repeat) ───────┘
```

### Agent 三大组件

| 组件 | 功能 | 例子 |
|------|------|------|
| **Policy** $\pi(a|s)$ | 选动作 | ε-greedy |
| **Value Function** $V(s)$ / $Q(s,a)$ | 评估好坏 | Q-table |
| **Model** (可选) | 预测未来 | 转移概率 |

> 💡 **Q-Learning = Value Based + Model Free**

运行下面的代码，可视化 Agent 分类 👇

In [ ]:
# Agent 分类可视化 | Agent taxonomy visualization
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')

categories = [
    ('Value Based', '❌ implicit', '✅', 'Optional', '#3498db'),
    ('Policy Based', '✅', '❌', 'Optional', '#e74c3c'),
    ('Actor-Critic', '✅ actor', '✅ critic', 'Optional', '#9b59b6'),
    ('Model Free', 'π and/or V/Q', '', '❌', '#f39c12'),
    ('Model Based', 'π and/or V/Q', '', '✅', '#2ecc71'),
]

table_data = [['Type', 'Policy', 'Value Fn', 'Model']]
for name, p, v, m, _ in categories:
    table_data.append([name, p, v, m])

table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

for j in range(4):
    table[0, j].set_facecolor('#34495e')
    table[0, j].set_text_props(color='white', fontweight='bold')
for i, (_, _, _, _, color) in enumerate(categories, 1):
    table[i, 0].set_facecolor(color)
    table[i, 0].set_text_props(color='white', fontweight='bold')

ax.set_title('Agent Classification (Q-Learning = Value Based + Model Free)',
             fontsize=13, fontweight='bold', pad=20)
plt.show()

### 🧠 练习 1: RL 概念判断

回答以下问题，然后运行代码检查答案：

1. RL 是监督学习的子类吗？
2. Policy 在 Agent 还是 Environment 中？
3. Reward 由谁提供？

In [ ]:
# 练习 1: 填入你的答案 | Exercise 1: Fill in your answers
# 改成 True 或 False | Change to True or False
my_answers = {
    "RL is a subset of supervised learning": False,   # ← 改这里
    "Policy is in the Agent": True,                    # ← 改这里
    "Reward comes from Environment": True,             # ← 改这里
}

# --- 答案检查 | Answer check ---
correct = {
    "RL is a subset of supervised learning": False,
    "Policy is in the Agent": True,
    "Reward comes from Environment": True,
}
for q, a in my_answers.items():
    status = "✅" if a == correct[q] else "❌"
    print(f"{status} {q}: your={a}, correct={correct[q]}")

## 2. MDP 与 Bellman 方程 (MDP & Bellman Equation) — Week 2

### 从 Week 1 到 Week 2 的过渡
> Week 1 说了"RL 有 Agent、Environment、Reward"，但怎么用**数学语言**精确描述？→ **MDP**

### MDP 五元组

$\langle S, A, P, R, \gamma \rangle$

| 符号 | 含义 | 说明 |
|------|------|------|
| $S$ | 状态集 (State Space) | 所有可能的状态 |
| $A$ | 动作集 (Action Space) | 所有可能的动作 |
| $P(s' \mid s, a)$ | 转移概率 | 状态转移是**随机的** |
| $R(s, a)$ | 奖励函数 | 即时反馈 |
| $\gamma$ | 折扣因子 | $0 \le \gamma < 1$ |

### Bellman 方程 (Q-Learning 版)

$$Q(s, a) = R + \gamma \max_{a'} Q(s', a')$$

### ★★★ Q-Learning 更新规则 (Midterm Slide 6 必考)

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ R + \gamma \max_{a'} Q(s', a') - Q(s, a) \Big]$$

运行下面的代码，交互式计算 Q-Learning 更新 👇

In [ ]:
# 交互式 Q-Learning 更新计算器 | Interactive Q-Learning update calculator
# ★ 修改这些值来练习不同场景 | Modify these values to practice

Q_current = 2.0    # 当前 Q(s,a) | Current Q-value
alpha = 0.1        # 学习率 | Learning rate
reward = 1.0       # 即时奖励 | Immediate reward
gamma = 0.9        # 折扣因子 | Discount factor
Q_next_max = 5.0   # max Q(s',a') | Max Q-value of next state

# --- 计算过程 | Calculation ---
td_target = reward + gamma * Q_next_max
td_error = td_target - Q_current
Q_new = Q_current + alpha * td_error

print("=" * 50)
print("Q-Learning Update Step-by-Step")
print("=" * 50)
print(f"Step 1: TD Target  = R + γ × max Q(s',a')")
print(f"                   = {reward} + {gamma} × {Q_next_max}")
print(f"                   = {td_target}")
print(f"")
print(f"Step 2: TD Error   = Target - Q(s,a)")
print(f"                   = {td_target} - {Q_current}")
print(f"                   = {td_error}")
print(f"")
print(f"Step 3: Update     = Q(s,a) + α × TD Error")
print(f"                   = {Q_current} + {alpha} × {td_error}")
print(f"                   = {Q_new}")
print(f"")
print(f"📊 Q(s,a): {Q_current} → {Q_new}")
print(f"   (向 TD Target {td_target} 靠近了 {abs(Q_new - Q_current):.2f})")

### 🧠 练习 2: ε-Greedy 概率计算

有 4 个动作，$\epsilon = 0.2$，Q 值分别为 $Q(s, a_1) = 1.5, Q(s, a_2) = 3.0, Q(s, a_3) = 2.0, Q(s, a_4) = 0.5$。

**问题：** 每个动作被选择的概率是多少？

先自己算，然后运行代码验证 👇

In [ ]:
# ε-Greedy 概率计算 | ε-Greedy probability calculation
epsilon = 0.2
q_values = {'a1': 1.5, 'a2': 3.0, 'a3': 2.0, 'a4': 0.5}
n_actions = len(q_values)

# 找到 greedy action | Find greedy action
greedy_action = max(q_values, key=q_values.get)
print(f"Greedy action: {greedy_action} (Q = {q_values[greedy_action]})")
print()

# 计算每个动作的概率 | Calculate probability for each action
print(f"{'Action':<8} {'Q-value':<10} {'Probability':<15} {'Calculation'}")
print("-" * 55)
total = 0
for action, q in q_values.items():
    if action == greedy_action:
        prob = (1 - epsilon) + epsilon / n_actions
        calc = f"(1-{epsilon}) + {epsilon}/{n_actions} = {prob:.2f}"
    else:
        prob = epsilon / n_actions
        calc = f"{epsilon}/{n_actions} = {prob:.2f}"
    total += prob
    marker = " ← greedy" if action == greedy_action else ""
    print(f"{action:<8} {q:<10} {prob:<15.2f} {calc}{marker}")

print(f"\nTotal probability: {total:.2f} ✅" if abs(total - 1.0) < 1e-9 else f"\nTotal: {total} ❌")

# 可视化 | Visualization
fig, ax = plt.subplots(figsize=(8, 4))
actions = list(q_values.keys())
probs = [epsilon / n_actions] * n_actions
greedy_idx = list(q_values.values()).index(max(q_values.values()))
probs[greedy_idx] += (1 - epsilon)
colors = ['#95a5a6'] * n_actions
colors[greedy_idx] = '#e74c3c'
bars = ax.bar(actions, probs, color=colors, edgecolor='black', linewidth=0.5)
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{p:.2f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('P(action)')
ax.set_title(f'ε-Greedy Action Probabilities (ε={epsilon})')
ax.set_ylim(0, 1.0)
plt.show()

## 3. Q-Learning vs SARSA — CliffWalking 深入 (Midterm Slide 4)

### 核心区别

| | Q-Learning | SARSA |
|---|---|---|
| **类型** | Off-policy (离策略) | On-policy (在策略) |
| **更新目标** | $\max_{a'} Q(s', a')$ | $Q(s', a'_{actual})$ |
| **CliffWalking** | 最短路径 (沿悬崖边) | 安全路径 (远离悬崖) |
| **原因** | max 忽略探索危险 | 考虑 ε-greedy 随机性 |

> ⚠️ **Midterm 讨论题：** 为什么 Q-Learning 收敛到最短路径而 SARSA 不同？
>
> → Q-Learning 是 off-policy，更新时用 max（假设未来总选最优），所以学到最短路径。
>
> → SARSA 是 on-policy，更新时考虑了实际探索行为（ε-greedy 可能走到悬崖边），所以学到更安全的路径。

运行下面的代码，模拟两种算法在 CliffWalking 上的行为 👇

In [ ]:
# Q-Learning vs SARSA on CliffWalking 模拟
# Simplified CliffWalking: 4x12 grid
rows, cols = 4, 12

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

paths = [
    ('Q-Learning (Off-Policy)', '#e74c3c',
     [(0,3),(1,3),(2,3),(3,3),(4,3),(5,3),(6,3),(7,3),(8,3),(9,3),(10,3),(11,3)],
     'Shortest path (along cliff)'),
    ('SARSA (On-Policy)', '#3498db',
     [(0,3),(0,2),(0,1),(0,0),(1,0),(2,0),(3,0),(4,0),(5,0),(6,0),
      (7,0),(8,0),(9,0),(10,0),(11,0),(11,1),(11,2),(11,3)],
     'Safe path (away from cliff)')
]

for ax_idx, (name, color, path, desc) in enumerate(paths):
    ax = axes[ax_idx]
    ax.set_xlim(-0.5, cols-0.5)
    ax.set_ylim(-0.5, rows-0.5)
    ax.set_aspect('equal')
    ax.invert_yaxis()

    for r in range(rows):
        for c in range(cols):
            fc = '#ecf0f1'
            label = ''
            if r == 3 and c == 0:
                fc, label = '#2ecc71', 'S'
            elif r == 3 and c == 11:
                fc, label = '#f1c40f', 'G'
            elif r == 3 and 1 <= c <= 10:
                fc, label = '#e74c3c', 'X'
            rect = mpatches.FancyBboxPatch((c-0.45, r-0.45), 0.9, 0.9,
                boxstyle="round,pad=0.02", facecolor=fc, edgecolor='#bdc3c7', linewidth=0.5)
            ax.add_patch(rect)
            if label:
                ax.text(c, r, label, ha='center', va='center', fontsize=10,
                        fontweight='bold', color='white' if label=='X' else '#2c3e50')

    for i in range(len(path)-1):
        x1, y1 = path[i]
        x2, y2 = path[i+1]
        ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2.5, alpha=0.8))

    ax.set_title(f'{name}\n{desc}', fontsize=11, fontweight='bold', color=color)
    ax.set_xticklabels([])
    ax.set_yticklabels([])

plt.suptitle('Q-Learning vs SARSA on CliffWalking', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 Key insight:")
print("  Q-Learning (off-policy): max ignores exploration danger → shortest but risky path")
print("  SARSA (on-policy): considers ε-greedy randomness → safer path away from cliff")

### 🧠 练习 3: Q 表初始化

**问题：** 终止状态的 Q 值为什么必须设为 0？

> 提示：终止状态没有"下一步"，$Q(s_{terminal}, a) = 0$ 对所有 $a$。

**试一试：** 修改下面代码中的初始化方式，观察对探索行为的影响 👇

In [ ]:
# Q-Table 初始化对比 | Q-Table initialization comparison
np.random.seed(42)
n_states, n_actions = 5, 4
state_labels = ['s0', 's1', 's2', 's3', 's_T (terminal)']

# ★ 试一试: 取消注释不同的初始化方式 | Try different initializations
# 方式 1: 零初始化 (conservative)
qtable = np.zeros((n_states, n_actions))

# 方式 2: 随机初始化 (moderate exploration)
# qtable = np.random.uniform(-0.5, 0.5, (n_states, n_actions))

# 方式 3: 乐观初始化 (aggressive exploration)
# qtable = np.ones((n_states, n_actions)) * 5.0

# ⚠️ 终止状态必须为 0 | Terminal state must be 0
qtable[-1, :] = 0

print("Q-Table (rows=states, cols=actions):")
print(f"{'State':<15} {'a0':>8} {'a1':>8} {'a2':>8} {'a3':>8}")
print("-" * 50)
for i, label in enumerate(state_labels):
    vals = ' '.join(f'{v:8.2f}' for v in qtable[i])
    marker = " ← must be 0!" if i == n_states-1 else ""
    print(f"{label:<15} {vals}{marker}")

## 4. Gymnasium API 回顾 (Gymnasium Review) — Week 3

### 从 Week 2 到 Week 3 的过渡
> 有了数学模型和算法，但怎么**实际运行**？→ **Gymnasium**

### Gymnasium 定义 (Midterm Slide 7 必考)

> Gymnasium is a framework for creating RL environments with a **standard interface** such that various RL algorithms/agents can be applied to the environment in a standard way.

### 核心 API

```python
env = gym.make("CliffWalking-v0")
state, info = env.reset()                                    # 重置
next_state, reward, terminated, truncated, info = env.step(action)  # 执行
env.close()                                                  # 关闭
```

### Wrapper (Midterm Slide 8 必考)

> Wrapper = 在**不修改底层代码**的情况下修改现有环境

运行下面的代码，练习 Gymnasium API 👇

In [ ]:
# Gymnasium API 演示 | Gymnasium API demo
try:
    import gymnasium as gym

    env = gym.make("CliffWalking-v0")
    print("=== Gymnasium CliffWalking-v0 ===")
    print(f"Observation space: {env.observation_space}")
    print(f"  n_states = {env.observation_space.n}")
    print(f"Action space: {env.action_space}")
    print(f"  n_actions = {env.action_space.n}")
    print(f"  Actions: 0=Up, 1=Right, 2=Down, 3=Left")
    print()

    # 运行一个 episode | Run one episode
    state, info = env.reset()
    print(f"reset() → state={state}, info={info}")

    # 执行几步 | Take a few steps
    for step in range(3):
        action = env.action_space.sample()  # 随机动作
        next_state, reward, terminated, truncated, info = env.step(action)
        print(f"step({action}) → state={next_state}, reward={reward}, "
              f"terminated={terminated}, truncated={truncated}")
        if terminated or truncated:
            break
        state = next_state

    env.close()
    print("\n✅ Gymnasium API works!")

except ImportError:
    print("⚠️ gymnasium not installed. Run: uv add gymnasium")
    print("\nAPI summary:")
    print("  env.reset()  → (state, info)")
    print("  env.step(a)  → (next_state, reward, terminated, truncated, info)")
    print("  env.close()  → cleanup")

### 🧠 练习 4: 自定义环境 Checklist

创建自定义 Gymnasium 环境需要实现哪些方法？

运行代码检查你的答案 👇

In [ ]:
# 自定义环境必须实现的方法 | Required methods for custom env
required_methods = {
    "__init__()": "定义 action_space 和 observation_space",
    "reset()": "返回 (initial_state, info)",
    "step(action)": "返回 (next_state, reward, terminated, truncated, info)",
    "render()": "可视化 (可选, 常用 Pygame)",
}

print("Custom Gymnasium Environment Checklist:")
print("=" * 55)
for method, desc in required_methods.items():
    print(f"  ✅ {method:<20} → {desc}")

print("\n⚠️ Wrapper: 不修改底层代码，修改环境行为")
print("   用法: env = MyWrapper(base_env)")

## 5. Stable-Baselines3 & DQN — Week 4-5

### SB3 定义 (Midterm Slide 9 必考)

> Stable-baselines3 is a set of **reliable** RL algorithm implementations.

关键特性:
- **Vectorized environments** — 同时运行多个环境副本 → 加速训练
- **Callbacks** — 自定义代码（监控、自动保存、进度条）

### 从 Q-Table 到 DQN 的演进

| | Tabular Q-Learning | DQN |
|---|---|---|
| Q值存储 | 表格 | 神经网络 |
| 状态空间 | 有限、已知 | 可连续/巨大 |
| 前提 | S 和 A 都已知有限 | 只需 A 已知 |
| 泛化 | ❌ 无 | ✅ 可泛化 |

### DQN 核心组件
1. **Q-Network** — 用神经网络逼近 Q 值
2. **Target Network** — 稳定训练目标（每 N 步同步）
3. **Replay Buffer** — 打破样本相关性
4. **ε-Greedy Decay** — 从探索到利用

运行下面的代码，体验 SB3 + DQN 训练 👇

In [ ]:
# SB3 DQN 快速训练演示 | SB3 DQN quick training demo
try:
    import gymnasium as gym
    from stable_baselines3 import DQN
    from stable_baselines3.common.evaluation import evaluate_policy

    env = gym.make("CartPole-v1")
    print("Training DQN on CartPole-v1...")
    print("(This may take ~30 seconds)\n")

    model = DQN("MlpPolicy", env, learning_rate=1e-3,
                buffer_size=50000, learning_starts=500,
                batch_size=64, gamma=0.99, verbose=0)

    # 训练前评估 | Evaluate before training
    mean_before, std_before = evaluate_policy(model, env, n_eval_episodes=5)
    print(f"Before training: reward = {mean_before:.1f} ± {std_before:.1f}")

    # 训练 | Train
    model.learn(total_timesteps=10000)

    # 训练后评估 | Evaluate after training
    mean_after, std_after = evaluate_policy(model, env, n_eval_episodes=5)
    print(f"After training:  reward = {mean_after:.1f} ± {std_after:.1f}")
    print(f"\n📈 Improvement: {mean_before:.1f} → {mean_after:.1f}")

    env.close()

except ImportError:
    print("⚠️ stable-baselines3 not installed. Run: uv add stable-baselines3")
    print("\nSB3 usage pattern:")
    print('  model = DQN("MlpPolicy", env, verbose=1)')
    print("  model.learn(total_timesteps=10000)")
    print("  action, _ = model.predict(obs, deterministic=True)")

## 6. 综合对比速查 (Comprehensive Comparison)

### 三大 ML 范式

| | Supervised | Unsupervised | **Reinforcement** |
|---|---|---|---|
| Feedback | Labels | None | **Reward signal** |
| Goal | Learn mapping | Find structure | **Maximize reward** |
| Exploration? | No | No | **Yes** |

### Agent vs Environment 职责 (Quiz 2 Q15)

| 组件 | 位置 | 功能 |
|------|------|------|
| Policy $\pi$ | **Agent** | 选动作 |
| Value $V/Q$ | **Agent** | 评估 |
| Transition $P$ | **Environment** | 状态转移 |
| Reward $R$ | **Environment** | 反馈 |

### ⚠️ 常见陷阱

| 陷阱 | 正确理解 | 来源 |
|------|----------|------|
| "Temporal Distance" | ❌ 错误术语，正确是 "Temporal **Difference**" | Quiz 2 Q14 |
| MDP 转移是确定性的 | ❌ 是**随机的** $P(s'|s,a)$ | Quiz 1 Q3 |
| Greedy = 最大化总回报 | ❌ Greedy = 即时奖励优先 | Quiz 2 Q12 |
| Q-Learning 只需知道 $A$ | ❌ 需要**同时**知道 $S$ 和 $A$ | Quiz 2 Q13 |

## 7. 试一试 (Try It Yourself!)

### 综合练习: 完整 Q-Learning 训练

修改超参数，观察对训练效果的影响：
- `alpha`: 学习率 (试试 0.01 vs 0.5)
- `gamma`: 折扣因子 (试试 0.5 vs 0.99)
- `epsilon_decay`: 衰减速度 (试试 0.99 vs 0.999)

In [ ]:
# 完整 Q-Learning 训练 | Full Q-Learning training
# ★ 修改这些超参数 | Modify these hyperparameters
alpha = 0.1           # 学习率 | Learning rate
gamma = 0.99          # 折扣因子 | Discount factor
epsilon = 1.0         # 初始探索率 | Initial exploration
epsilon_decay = 0.995 # 衰减率 | Decay rate
epsilon_min = 0.05    # 最小探索率 | Minimum exploration
num_episodes = 500    # 训练回合数 | Training episodes

try:
    import gymnasium as gym
    env = gym.make("CliffWalking-v0")
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    qtable = np.zeros((n_states, n_actions))

    rewards_per_episode = []

    for ep in range(num_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False

        while not done:
            # ε-greedy
            if np.random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(qtable[state])

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # Q-Learning update
            qtable[state][action] += alpha * (
                reward + gamma * np.max(qtable[next_state]) - qtable[state][action]
            )
            state = next_state
            total_reward += reward

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_per_episode.append(total_reward)

    env.close()

    # 可视化学习曲线 | Visualize learning curve
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    # 左: 奖励曲线 | Left: reward curve
    window = 20
    smoothed = np.convolve(rewards_per_episode, np.ones(window)/window, mode='valid')
    ax1.plot(smoothed, color='#3498db', linewidth=1.5)
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward (smoothed)')
    ax1.set_title(f'Learning Curve (α={alpha}, γ={gamma})')
    ax1.grid(True, alpha=0.3)

    # 右: 最终策略 | Right: final policy
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    policy_grid = np.zeros((4, 12), dtype=int)
    for s in range(n_states):
        r, c = divmod(s, 12)
        policy_grid[r][c] = np.argmax(qtable[s])

    ax2.set_xlim(-0.5, 11.5)
    ax2.set_ylim(-0.5, 3.5)
    ax2.set_aspect('equal')
    ax2.invert_yaxis()
    for r in range(4):
        for c in range(12):
            fc = '#ecf0f1'
            if r == 3 and c == 0: fc = '#2ecc71'
            elif r == 3 and c == 11: fc = '#f1c40f'
            elif r == 3 and 1 <= c <= 10: fc = '#e74c3c'
            rect = mpatches.FancyBboxPatch((c-0.45, r-0.45), 0.9, 0.9,
                boxstyle="round,pad=0.02", facecolor=fc, edgecolor='#bdc3c7', linewidth=0.5)
            ax2.add_patch(rect)
            if not (r == 3 and 1 <= c <= 10):
                ax2.text(c, r, arrows[policy_grid[r][c]], ha='center', va='center', fontsize=12)
    ax2.set_title('Learned Policy')
    ax2.set_xticklabels([])
    ax2.set_yticklabels([])

    plt.tight_layout()
    plt.show()

    print(f"\nFinal avg reward (last 50 eps): {np.mean(rewards_per_episode[-50:]):.1f}")

except ImportError:
    print("⚠️ gymnasium not installed")

## 8. 考试检查清单 (Exam Checklist)

### ✍️ 必须能写的
- [ ] **画 Agent-Environment 交互图** (Slide 5)
- [ ] **写 Q-Learning 更新公式** + 每个变量含义 (Slide 6)
- [ ] 定义 Gymnasium (Slide 7)
- [ ] 定义 Gymnasium Wrapper (Slide 8)
- [ ] 定义 Stable-Baselines3 + 两个关键特性 (Slide 9)

### 🧠 必须能回答的
- [ ] RL 与监督/无监督学习的区别
- [ ] 马尔可夫性质的定义
- [ ] $V(s)$ vs $Q(s,a)$ 的区别
- [ ] Q-Learning vs SARSA 的区别和 CliffWalking 结果
- [ ] Q 表初始化对收敛的影响
- [ ] 终止状态 Q 值为什么设为 0
- [ ] TD ≠ Temporal Distance (是 Temporal **Difference**)

### 📊 必须能对比的
- [ ] Q-Learning vs SARSA (off-policy vs on-policy)
- [ ] Tabular Q vs DQN (表格 vs 神经网络)
- [ ] Value Based vs Policy Based vs Actor-Critic
- [ ] Model Free vs Model Based

---

> 📚 **相关资料:**
> - [概念速查](week6_midterm_review_cheatsheet.md)
> - [数学公式](week6_midterm_review_math.md)
> - [代码参考](week6_midterm_review_code.md)
> - [故事线](week6_midterm_review_storyline.md)